# Module 06-1: Pydantic 모델로 출력 스키마 정의

## 학습 목표
- `BaseModel`과 `Field`로 데이터 스키마를 정의할 수 있다
- `Field` 제약조건(ge, le, pattern, min_length)을 사용하여 데이터를 검증할 수 있다
- `Enum`으로 허용값을 제한할 수 있다
- `field_validator`로 커스텀 검증 로직을 추가할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/06-structured-output.md` 섹션 2.1

## 개념 요약

### Pydantic이란?

Python의 데이터 검증 라이브러리입니다. `BaseModel`을 상속하여 클래스를 정의하면, 생성 시 자동으로 타입과 제약조건을 검증합니다.

### Field 옵션 정리

| 옵션 | 용도 | 예시 |
|------|------|------|
| `description` | 필드 설명 (LLM 힌트) | `Field(description="버그 설명")` |
| `min_length` | 문자열 최소 길이 | `Field(min_length=1)` |
| `max_length` | 문자열 최대 길이 | `Field(max_length=500)` |
| `ge` / `le` | 숫자 범위 (이상/이하) | `Field(ge=0, le=1.0)` |
| `pattern` | 정규식 패턴 매칭 | `Field(pattern=r"^(HIGH|MEDIUM|LOW)$")` |
| `default` | 기본값 | `Field(default="MEDIUM")` |
| `default_factory` | 기본값 팩토리 | `Field(default_factory=list)` |

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from pydantic import BaseModel, Field, field_validator, ValidationError
from enum import Enum
from typing import Optional

print("환경 설정 완료")

## 실습 1: 기본 BaseModel과 Field 제약조건

`BugReport` 모델을 정의하고, 올바른 데이터와 잘못된 데이터의 검증 동작을 확인합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): BaseModel을 상속하여 BugReport 클래스를 정의하세요.
#               각 필드에 Field()로 제약조건을 추가하세요.
#
# 힌트 2 (키워드): class BugReport(BaseModel):, ge=1, min_length=1, pattern=r"^(HIGH|MEDIUM|LOW)$"
#
# 힌트 3 (거의 정답):
#   class BugReport(BaseModel):
#       line: int = Field(description="버그가 있는 라인 번호", ge=1)
#       issue: str = Field(description="버그 설명", min_length=1)
#       severity: str = Field(description="심각도", pattern=r"^(HIGH|MEDIUM|LOW)$")

# 1-1. BugReport 모델 정의
class BugReport(BaseModel):
    """버그 리포트 스키마."""
    pass  # TODO: line, issue, severity 필드 정의

In [ ]:
# 1-2. 올바른 데이터로 인스턴스 생성
valid_bug = BugReport(line=5, issue="null check missing", severity="HIGH")
print("올바른 데이터:")
print(valid_bug.model_dump())

print()

# 1-3. 잘못된 데이터로 인스턴스 생성 (ValidationError 발생 확인)
print("잘못된 데이터 검증:")
try:
    bad_bug = BugReport(line=-1, issue="", severity="CRITICAL")
    print("검증이 통과되었습니다 (기대하지 않은 결과)")
except ValidationError as e:
    print(f"ValidationError 발생: {len(e.errors())}개 오류")
    for error in e.errors():
        print(f"  - {error['loc'][0]}: {error['msg']}")

In [ ]:
# 검증 셀
# 올바른 데이터 검증
test_bug = BugReport(line=1, issue="test issue", severity="MEDIUM")
assert test_bug.line == 1, "line 필드가 올바르게 설정되지 않았습니다."
assert test_bug.issue == "test issue", "issue 필드가 올바르게 설정되지 않았습니다."
assert test_bug.severity == "MEDIUM", "severity 필드가 올바르게 설정되지 않았습니다."

# 잘못된 데이터 검증 확인
try:
    BugReport(line=0, issue="test", severity="INVALID")  # line >= 1, severity 패턴
    assert False, "ValidationError가 발생해야 합니다."
except ValidationError:
    pass  # 정상

try:
    BugReport(line=1, issue="", severity="HIGH")  # issue 빈 문자열 금지
    assert False, "ValidationError가 발생해야 합니다."
except ValidationError:
    pass  # 정상

print("실습 1 완료!")

## 실습 2: Enum으로 허용값 제한

문자열 대신 `Enum`을 사용하여 허용되는 값을 명시적으로 제한합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): str과 Enum을 동시에 상속하는 클래스를 만들어 문자열처럼 사용할 수 있게 합니다.
#               TestResult 모델에서 status와 severity 필드에 Enum 타입을 사용하세요.
#
# 힌트 2 (키워드): class Severity(str, Enum):, class TestStatus(str, Enum):, TestResult(BaseModel)
#
# 힌트 3 (거의 정답):
#   class Severity(str, Enum):
#       HIGH = "HIGH"
#       MEDIUM = "MEDIUM"
#       LOW = "LOW"
#
#   class TestStatus(str, Enum):
#       PASS = "PASS"
#       FAIL = "FAIL"
#       BLOCKED = "BLOCKED"
#
#   class TestResult(BaseModel):
#       status: TestStatus
#       confidence: float = Field(ge=0.0, le=1.0)
#       reasoning: str = Field(min_length=1)
#       severity: Severity = Field(default=Severity.MEDIUM)

# 2-1. Severity Enum 정의
class Severity(str, Enum):
    pass  # TODO: HIGH, MEDIUM, LOW 값 추가

# 2-2. TestStatus Enum 정의
class TestStatus(str, Enum):
    pass  # TODO: PASS, FAIL, BLOCKED 값 추가

# 2-3. TestResult 모델 정의
class TestResult(BaseModel):
    """테스트 결과 스키마."""
    pass  # TODO: status, confidence, reasoning, severity 필드 정의

In [ ]:
# 2-4. 테스트
result = TestResult(
    status=TestStatus.PASS,
    confidence=0.95,
    reasoning="모든 테스트 케이스 통과"
)
print("TestResult 생성:")
print(result.model_dump())
print(f"\nseverity 기본값: {result.severity}")

# 잘못된 값으로 테스트
try:
    bad_result = TestResult(status="INVALID_STATUS", confidence=0.5, reasoning="테스트")
    print("\n오류: ValidationError가 발생해야 합니다")
except ValidationError as e:
    print(f"\n올바르게 ValidationError 발생: {e.errors()[0]['msg']}")

In [ ]:
# 검증 셀
# Enum 값 확인
assert hasattr(Severity, 'HIGH'), "Severity.HIGH가 없습니다."
assert hasattr(Severity, 'MEDIUM'), "Severity.MEDIUM이 없습니다."
assert hasattr(Severity, 'LOW'), "Severity.LOW가 없습니다."
assert hasattr(TestStatus, 'PASS'), "TestStatus.PASS가 없습니다."
assert hasattr(TestStatus, 'FAIL'), "TestStatus.FAIL이 없습니다."
assert hasattr(TestStatus, 'BLOCKED'), "TestStatus.BLOCKED가 없습니다."

# TestResult 검증
test_result = TestResult(status="PASS", confidence=0.8, reasoning="테스트 통과")
assert test_result.status == TestStatus.PASS, "status가 TestStatus.PASS여야 합니다."

try:
    TestResult(status="PASS", confidence=1.5, reasoning="테스트")  # confidence > 1.0
    assert False, "ValidationError가 발생해야 합니다."
except ValidationError:
    pass

print("실습 2 완료!")

## 실습 3: field_validator로 커스텀 검증

`@field_validator`로 비즈니스 로직을 검증하는 커스텀 유효성 검사기를 추가합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): @field_validator 데코레이터와 @classmethod를 함께 사용합니다.
#               validator는 값을 받아 변환하거나 예외를 발생시킵니다.
#
# 힌트 2 (키워드): @field_validator("field_name"), @classmethod, round(v, 2), [f for f in v if f.strip()]
#
# 힌트 3 (거의 정답):
#   class CodeAnalysis(BaseModel):
#       affected_files: list[str] = Field(min_length=1)
#       confidence: float = Field(ge=0.0, le=1.0)
#       root_cause: str = Field(min_length=1)
#
#       @field_validator("confidence")
#       @classmethod
#       def round_confidence(cls, v: float) -> float:
#           return round(v, 2)
#
#       @field_validator("affected_files")
#       @classmethod
#       def validate_file_paths(cls, v: list[str]) -> list[str]:
#           return [f for f in v if f.strip()]

class CodeAnalysis(BaseModel):
    """코드 분석 결과 스키마."""
    affected_files: list[str] = Field(min_length=1, description="영향받는 파일 목록")
    confidence: float = Field(ge=0.0, le=1.0, description="분석 신뢰도")
    root_cause: str = Field(min_length=1, description="근본 원인")

    # TODO: confidence를 소수점 2자리로 반올림하는 validator 추가

    # TODO: affected_files에서 빈 문자열을 제거하는 validator 추가

In [ ]:
# 3-2. 테스트
analysis = CodeAnalysis(
    affected_files=["src/main.py", "", "src/utils.py"],  # 빈 문자열 포함
    confidence=0.85678,  # 소수점 많음
    root_cause="Missing null check"
)

print("CodeAnalysis 생성:")
print(f"  confidence: {analysis.confidence} (0.85678 -> 소수점 2자리로 반올림)")  
print(f"  affected_files: {analysis.affected_files} (빈 문자열 제거됨)")
print(f"  root_cause: {analysis.root_cause}")

In [ ]:
# 검증 셀
test_analysis = CodeAnalysis(
    affected_files=["main.py", "", "utils.py"],
    confidence=0.12345,
    root_cause="테스트 원인"
)

assert test_analysis.confidence == 0.12, f"confidence가 0.12여야 합니다. 현재: {test_analysis.confidence}\nfield_validator('confidence')를 구현하세요."
assert "" not in test_analysis.affected_files, f"빈 문자열이 제거되지 않았습니다. 현재: {test_analysis.affected_files}\nfield_validator('affected_files')를 구현하세요."
assert len(test_analysis.affected_files) == 2, f"빈 문자열 제거 후 2개 파일이 남아야 합니다. 현재: {test_analysis.affected_files}"
print("실습 3 완료!")

## 실습 4: 중첩 모델

복잡한 출력 구조를 위해 Pydantic 모델을 중첩하여 사용합니다.

In [ ]:
# 4-1. 중첩 모델 정의 (완성된 예시)
class SuggestedFix(BaseModel):
    """수정 제안."""
    description: str = Field(description="수정 방법 설명")
    code_snippet: str = Field(default="", description="수정 코드 예시")

class BugAnalysis(BaseModel):
    """단일 버그 분석 결과."""
    title: str = Field(min_length=1, description="버그 제목")
    severity: Severity = Field(description="심각도")
    description: str = Field(min_length=10, description="상세 설명")
    suggested_fix: SuggestedFix = Field(description="수정 제안")  # 중첩 모델!

class BugReportAnalysis(BaseModel):
    """전체 버그 리포트 분석 결과."""
    total_bugs: int = Field(ge=0, description="발견된 총 버그 수")
    bugs: list[BugAnalysis] = Field(default_factory=list)  # 중첩 모델 리스트!
    summary: str = Field(min_length=1, description="분석 요약")

# 4-2. 테스트
report = BugReportAnalysis(
    total_bugs=1,
    bugs=[
        BugAnalysis(
            title="SQL Injection 취약점",
            severity=Severity.HIGH,
            description="사용자 입력을 직접 SQL 쿼리에 삽입하여 SQL Injection 공격에 취약합니다.",
            suggested_fix=SuggestedFix(
                description="파라미터화된 쿼리 사용",
                code_snippet="db.execute('SELECT * FROM users WHERE id = ?', (user_id,))"
            )
        )
    ],
    summary="1개의 심각한 보안 취약점 발견"
)

print("BugReportAnalysis 생성:")
print(f"  총 버그 수: {report.total_bugs}")
print(f"  첫 번째 버그: {report.bugs[0].title}")
print(f"  심각도: {report.bugs[0].severity}")
print(f"  수정 방법: {report.bugs[0].suggested_fix.description}")

In [ ]:
# 검증 셀
assert report.total_bugs == 1, "total_bugs가 1이어야 합니다."
assert len(report.bugs) == 1, "bugs 리스트에 1개 항목이 있어야 합니다."
assert isinstance(report.bugs[0], BugAnalysis), "bugs 항목이 BugAnalysis 인스턴스이어야 합니다."
assert isinstance(report.bugs[0].suggested_fix, SuggestedFix), "suggested_fix가 SuggestedFix 인스턴스이어야 합니다."
print("실습 4 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **BaseModel + Field**: 스키마 정의와 제약조건 설정
2. **Field 제약조건**: `ge`, `le`, `min_length`, `max_length`, `pattern`, `default`, `default_factory`
3. **Enum**: 허용값을 명시적으로 제한하여 타입 안전성 확보
4. **field_validator**: 커스텀 검증 로직 (값 변환, 필터링)
5. **중첩 모델**: 복잡한 데이터 구조를 계층적으로 표현

**다음 실습**: `02_structured_output.ipynb` - `with_structured_output()`으로 LLM 응답 강제